In [ ]:
BATCH_SIZE = 32

SEQ_LEN = 20 
EMBED_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2

FF_DIM = 256

DROPOUT = 0.3

EPOCHS = 10

LR = 3e-4
WEIGHT_DECAY = 1e-4

In [76]:
df=pd.read_csv("poetry.csv") 
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(df,test_size=0.2,random_state=42)
val_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=42)

In [77]:
def clean(text):
    text = re.sub(r'["\'()\[\]{}]', '', text)
    text = re.sub(r'[.,!?;:]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [78]:
train_poems = [clean(p) for p in train_df["content"]]
val_poems   = [clean(p) for p in val_df["content"]]
test_poems  = [clean(p) for p in test_df["content"]]


train_text = "\n\n".join(train_poems)
train_tokens = train_text.split()
special_tokens = ["<PAD>", "<UNK>"]

vocab = special_tokens+sorted(set(train_tokens))

stoi = {word: idx for idx, word in enumerate(vocab)}
itos = {idx: word for idx, word in enumerate(vocab)}
vocab_size = len(vocab)

In [79]:
def encode(text):
    tokens = text.split()
    return [stoi.get(token, stoi["<UNK>"]) for token in tokens]

def decode(ids):
    return " ".join(itos[idx] for idx in ids)

In [80]:
def generate(model, prompt, max_new_tokens=50):
    model.eval()
    tokens = encode(prompt)
    x = tc.tensor(tokens, dtype=tc.long, device=device).unsqueeze(0)
    with tc.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -SEQ_LEN:]
            logits = model(x_cond)
            logits = logits[:, -1, :]
            next_token = tc.argmax(logits, dim=-1, keepdim=True)
            x = tc.cat([x, next_token], dim=1)
    generated_ids = x[0].tolist()
    return decode(generated_ids)

In [81]:
train_data = encode(" ".join(train_poems))
val_data = encode(" ".join(val_poems))
test_data = encode(" ".join(test_poems))

In [82]:
def train(model, train_loader, val_loader, epochs=10):

    criterion = nn.CrossEntropyLoss()
    optimizer = tc.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Training
        model.train()
        running_train_loss = 0.0

        train_bar = tqdm(train_loader,desc=f"Epoch {epoch+1}/{epochs} [Train]",leave=False)

        for x_batch, y_batch in train_bar:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits.reshape(-1, logits.size(-1)),y_batch.reshape(-1))

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        running_val_loss = 0.0

        val_bar = tqdm(val_loader,desc=f"Epoch {epoch+1}/{epochs} [Val]",leave=False)
        with tc.no_grad():
            for x_batch, y_batch in val_bar:

                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(x_batch)
                loss = criterion(logits.reshape(-1, logits.size(-1)),y_batch.reshape(-1))

                running_val_loss += loss.item()

                val_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    return train_losses, val_losses

In [83]:
def tok_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    with tc.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            preds = logits.argmax(dim=-1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return 100 * correct / total

In [84]:
class Dataset(tc.utils.data.Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]

        return (tc.tensor(x, dtype=tc.long),tc.tensor(y, dtype=tc.long))

In [85]:
class Positional(nn.Module):
    def __init__(self, max_seq_len, embed_dim):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)

    def forward(self, x):
        B, T = x.shape
        positions = tc.arange(T, device=x.device)
        positions = positions.unsqueeze(0).expand(B, T)

        return self.pos_embedding(positions)

In [86]:
train_dataset = Dataset(train_data, SEQ_LEN)
val_dataset   = Dataset(val_data, SEQ_LEN)
test_dataset  = Dataset(test_data, SEQ_LEN)

In [87]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [88]:
class Attention(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key   = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(K.size(-1))
        mask = tc.tril(tc.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = weights @ V
        return out

In [89]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList([Attention(embed_dim, head_dim)for _ in range(num_heads)])
        self.fc = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        out = tc.cat([head(x) for head in self.heads],dim=-1)
        out = self.fc(out)
        return out

In [90]:
class Forward(nn.Module):
    def __init__(self, embed_dim, ff_dim, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [91]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=DROPOUT):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = Forward(embed_dim=embed_dim,ff_dim=ff_dim,dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out = self.attn(x)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x

In [92]:
class GPT(nn.Module):
    def __init__(self,vocab_size,embed_dim,num_heads,num_layers,ff_dim,max_seq_len,dropout=DROPOUT):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size,embed_dim)
        self.pos_emb = Positional(max_seq_len,embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[DecoderBlock(embed_dim,num_heads,ff_dim,dropout)for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim,vocab_size)

    def forward(self, x):
        tok_embed = self.tok_emb(x)
        pos_embed = self.pos_emb(x)
        x = tok_embed + pos_embed
        x = self.dropout(x)
        x = self.blocks(x)
        x = self.norm(x)
        logits = self.lm_head(x)

        return logits

In [93]:
model = GPT(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    max_seq_len=SEQ_LEN,
    dropout=DROPOUT
).to(device)

In [94]:
train_losses, val_losses = train(model,train_loader,val_loader,epochs=10)

Epoch 1/10 | Train Loss: 7.1127 | Val Loss: 7.4153


Epoch 2/10 | Train Loss: 6.1034 | Val Loss: 7.4357


Epoch 3/10 | Train Loss: 5.1605 | Val Loss: 7.7887


Epoch 4/10 | Train Loss: 4.1682 | Val Loss: 8.4669


Epoch 5/10 | Train Loss: 3.6264 | Val Loss: 8.9230


Epoch 6/10 | Train Loss: 3.2801 | Val Loss: 9.3196


Epoch 7/10 | Train Loss: 2.9585 | Val Loss: 9.6286


Epoch 8/10 | Train Loss: 2.6851 | Val Loss: 9.9291


Epoch 9/10 | Train Loss: 2.4739 | Val Loss: 10.1244


Epoch 10/10 | Train Loss: 2.3233 | Val Loss: 10.2627


In [95]:
tc.save(model.state_dict(),"gpt.pth")

In [96]:
model.load_state_dict(tc.load("gpt.pth", map_location=device))
model.eval()

GPT(
  (tok_emb): Embedding(11167, 64)
  (pos_emb): Positional(
    (pos_embedding): Embedding(50, 64)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (blocks): Sequential(
    (0): DecoderBlock(
      (attn): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Attention(
            (query): Linear(in_features=64, out_features=16, bias=False)
            (key): Linear(in_features=64, out_features=16, bias=False)
            (value): Linear(in_features=64, out_features=16, bias=False)
          )
        )
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ff): Forward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): ReLU()
          (2): Linear(in_features=256, out_features=64, bias=True)
          (3): Dropout(p=0.3, inplace=False)
        )
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine

In [113]:
val_acc = tok_acc(model, val_loader)
print("GPT Accuracy : ",val_acc)

GPT Accuracy :  10.525185446838574


In [112]:
prompt = input("Give the starting phrase/word for the poem : ")

print(generate(model, prompt, max_new_tokens=40))

Love is not so much more for this song that I have been worth thinking on this tyranny To ungod this child again To ungod this child again To ungod this child again it could not be not be not be not


In [99]:
def generate_temp(model, prompt, max_new_tokens=50, temperature=1.0):
    model.eval()
    tokens = encode(prompt)
    x = tc.tensor(tokens, dtype=tc.long, device=device).unsqueeze(0)
    with tc.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -SEQ_LEN:]
            logits = model(x_cond)
            logits = logits[:, -1, :]
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = tc.multinomial(probs, num_samples=1)
            x = tc.cat([x, next_token], dim=1)
    return decode(x[0].tolist())

In [105]:
print("Temperature = 0.7")
print(generate_temp(model, prompt, 50, temperature=0.7))
print("\n\nTemperature = 1.0")
print(generate_temp(model, prompt, 50, temperature=1.0))
print("\n\nTemperature = 1.5")
print(generate_temp(model, prompt, 50, temperature=1.5))
temp=float(input("Custom Temperature : "))
print("\n\nTemperature = ",temp)
print(generate_temp(model, prompt, 50, temperature=temp))

Temperature = 0.7
Moon tell me lie I was I am thine my horse my heart invadeth Farewell thou hast engrossed Of myself myself myself myself myself myself myself in my heart I only thee and thee please thee I have I think thee and swoon Then shalt thou brokst not but lent And


Temperature = 1.0
Moon tell me very smooth pillows sweetest bed A gown Homespun dyed butternuts dark gold color Lost more finesse with the Pincke and purple silent haunches and holy season fit Chance or we court and soon And tomorrow maybe for me one anothers best tis imposture all And Januarys loud-seeming sun


Temperature = 1.5
Moon tell my breast I promised me beloved love my flame it wasted in laurel tree displayes O may not with thirsty eyes on Circes how bent me hand defacd The grandiose gestures Of other As Nature wept thinking on was fair Venus under her my dropping of Walsinghame Met you


Temperature =  0.5
Moon tell me thus Say nay say nay say nay But as thou leave to justify For nothing doth waste For

In [106]:
def generate_topk(model, prompt, max_new_tokens=50, k=20):
    model.eval()
    tokens = encode(prompt)
    x = tc.tensor(tokens, dtype=tc.long, device=device).unsqueeze(0)
    with tc.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -SEQ_LEN:]
            logits = model(x_cond)
            logits = logits[:, -1, :]
            values, indices = tc.topk(logits, k)
            probs = F.softmax(values, dim=-1)
            sampled = tc.multinomial(probs, num_samples=1)
            next_token = indices.gather(-1, sampled)
            x = tc.cat([x, next_token], dim=1)
    return decode(x[0].tolist())

In [108]:
print("Top-k = 5")
print(generate_topk(model, prompt, 50, k=5))
print("\n\nTop-k = 20")
print(generate_topk(model, prompt, 50, k=20))
print("\n\nTop-k = 50")
print(generate_topk(model, prompt, 50, k=50))
cust_k=int(input("Custom k : "))
print("\n\nTop-k = ",cust_k)
print(generate_topk(model, prompt, 50, k=cust_k))

Top-k = 5
Moon tell me thus Say nay say nay New yeare his sake Knowing worlds contracted to me and grame And wilt thou wilt thou leave my heart thus Say nay say nay New yeare Quenching the gasping furrowes thirst with rayne Like April shoure so stremes the trickling teares Adowne thy


Top-k = 20
Moon tell me the last Stark naked in mine eyes had forgotten In joy before Iam loth By sun may please she might see Falsehood is her heir Thyself content me Spare not that Paradise and hate Hate of sin grounded on sinful thoughts sold cheap what is cruellest and hastes


Top-k = 50
Moon tell me And the end of woe Studying inventions fine delight by her wits to entertain Oft turning others leaves thence want see Will whip her speakes And blesseth her naked glory strove To please you fayre Elisa Queene in one High cool wide from the slowly smoldering fire Of


Top-k =  30
Moon tell me I go seek some more from them led To hear unlike the dusk A pregnant pot Go wailing verse the infants of my hear

In [109]:
def generate_topp(model, prompt, max_new_tokens=50, p=0.8):
    model.eval()
    tokens = encode(prompt)
    x = tc.tensor(tokens, dtype=tc.long, device=device).unsqueeze(0)
    with tc.no_grad():
        for _ in range(max_new_tokens):
            x_cond = x[:, -SEQ_LEN:]
            logits = model(x_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            sorted_probs, sorted_indices = tc.sort(probs,descending=True)
            cumulative_probs = tc.cumsum(sorted_probs,dim=-1)
            mask = cumulative_probs > p
            mask[..., 1:] = mask[..., :-1].clone()
            mask[..., 0] = False
            sorted_probs[mask] = 0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1,keepdim=True)
            sampled = tc.multinomial(sorted_probs,num_samples=1)
            next_token = sorted_indices.gather(-1,sampled)
            x = tc.cat([x, next_token], dim=1)
    return decode(x[0].tolist())

In [110]:
print("Top-p = 0.8")
print(generate_topp(model, prompt, 50, p=0.8))
print("\n\nTop-p = 0.9")
print(generate_topp(model, prompt, 50, p=0.9))
print("\n\nTop-p = 0.95")
print(generate_topp(model, prompt, 50, p=0.95))
cust_p=float(input("Custom p : "))
print("\n\nTop-p = ",cust_p)
print(generate_topp(model, prompt, 50, p=cust_p))

Top-p = 0.8
Moon tell me thus Say nay And wilt thou leave me thus Say nay say nay say nay And wilt thou wilt thou subornd informer a true soul When true soul of true blood drops of this worlds soul looks like this tossing mew But madest my sprite live and belly


Top-p = 0.9
Moon tell her cloak aspiring minds Like rain of air Sound above the evening sitting on her beauty to this purpose but a torn or low steamboat whistle Finding a way mid mist on your brain captivd in golden cage O fool though he at fault on Sees ef all de


Top-p = 0.95
Moon tell me the life from the table Let me and me know beforehand And far I came To meet Though I shall tread Now the prouder grew out of my heart truly Be slow to hear and wore And so excellent Yet hath spent all to brydle love HOBBINOLL Colin


Top-p =  0.5
Moon tell me thus Say nay say nay say nay say nay And wilt thou weepst unkindly kind My lifes delight to love doth not be not knotted With hope that is not be growing beauties not be thy nest Co

In [111]:
prompts = [
    "Love is",
    "The moon",
    "In the night",
    "I remember"
]
cust_p=float(input("Custom p : "))
for prompt in prompts:
    print("="*80)
    print("Prompt:", prompt)
    print()
    print(generate_topp(model, prompt, 50, p=cust_p))
    print()

Prompt: Love is

Love is as waves continually As she my Damzell doth conceave Th importune suit the dayes doo weave Grey hairs upon my young maiden since liberty Ye old rag to sleep to sleep and by beautys fears Ah Desire Doth plunge my well-formed soul even in your tongue These worlds light And

Prompt: The moon

The moon has fled in the lamplight downed with flowers and flowers and light brown surprise Should not from the leader laughs low steamboat whistle Finding a shoreland Where gray rocks let fall Beneath the music plants and flowers and flowers shrivelled and give my head I did I have seen by

Prompt: In the night

In the night slur into dawn Hear the slow Naked on the steep soft lip Of the turf they went about the gorse flowers Wed summer merrily Merrily merrily Merrily merrily Merrily merrily Merrily merrily Merrily merrily shall live now Under the motor cars whirr by soon in hand light-headed Bacchus fruite is

Prompt: I remember

I remember plain Nor eat them go seek some o